# EURUSD Trading Analysis and Reinforcement Learning System
A comprehensive trading system combining traditional analysis with reinforcement learning.

## Table of Contents
1. Core Trading Components
2. Data Analysis and Visualization
3. Reinforcement Learning Implementation

In [45]:
# Core imports and configurations
import pandas as pd
import numpy as np
from datetime import datetime
import logging
import pandas_ta as ta
import plotly.express as px
import plotly.graph_objects as go
import tensorflow as tf
from collections import deque
import random
import os

# Configure logging
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('trading_sim.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

In [46]:
class TradingSimulator:
    """Core trading simulation engine"""
    
    def __init__(self, initial_balance=100, lot_size=0.1):
        """Initialize trading simulator with given parameters"""
        self.balance = initial_balance
        self.lot_size = lot_size  # 0.1 lots = 10,000 units
        self.pip_value = (10 * self.lot_size)  # For EURUSD, 1 standard lot = $10 per pip
        self.stop_loss_pips = 25
        self.current_trade = None
        self.trades_history = []
    
    def calculate_position_size(self):
        """Calculate position size based on risk management"""
        risk_per_trade = self.balance * 0.02  # 2% risk per trade
        risk_per_pip = risk_per_trade / self.stop_loss_pips
        return min(self.lot_size, round(risk_per_pip / 10, 2))  # Divide by $10 (1 standard lot pip value)
    
    def open_trade(self, entry_price, direction, timestamp):
        """Open a new trade position"""
        if self.current_trade is not None:
            return False
            
        stop_loss = entry_price - (self.stop_loss_pips * 0.0001) if direction == 'buy' else \
                    entry_price + (self.stop_loss_pips * 0.0001)
        
        position_size = self.calculate_position_size()
        
        self.current_trade = {
            'entry_time': timestamp,
            'entry_price': entry_price,
            'direction': direction,
            'stop_loss': stop_loss,
            'position_size': position_size,
            'status': 'open'
        }
        
        logger.info(f"Opened {direction} position: Entry={entry_price}, SL={stop_loss}, Size={position_size}")
        return True
    
    def close_trade(self, exit_price, timestamp, reason="signal"):
        """Close current trade and calculate profit/loss"""
        if self.current_trade is None:
            return
            
        # Calculate pip difference (0.0001 = 1 pip)
        pip_difference = (exit_price - self.current_trade['entry_price']) * 10000
        if self.current_trade['direction'] == 'sell':
            pip_difference = -pip_difference
            
        # Calculate profit: pips × pip_value (adjusted for position size)
        profit = pip_difference * self.current_trade['position_size'] * 10  # $10 per pip per standard lot
        self.balance += profit
        
        trade_result = {
            **self.current_trade,
            'exit_time': timestamp,
            'exit_price': exit_price,
            'profit': profit,
            'pips': pip_difference,
            'close_reason': reason
        }
        
        self.trades_history.append(trade_result)
        logger.info(f"Closed trade: Pips={pip_difference:.1f}, Profit=${profit:.2f}, Balance=${self.balance:.2f}")
        
        self.current_trade = None
        return trade_result
    
    def check_stop_loss(self, current_price, timestamp):
        """Check if stop loss has been hit"""
        if self.current_trade is None:
            return False
            
        if self.current_trade['direction'] == 'buy':
            if current_price <= self.current_trade['stop_loss']:
                self.close_trade(current_price, timestamp, "stop_loss")
                return True
        else:  # sell
            if current_price >= self.current_trade['stop_loss']:
                self.close_trade(current_price, timestamp, "stop_loss")
                return True
        return False
    
    def process_signals(self, m5_data, h1_data):
        """Process trading signals and execute trades"""
        results = []
        
        for idx in m5_data.index:
            current_price = m5_data.loc[idx, 'close']
            current_time = idx
            
            # Check stop loss first
            if self.check_stop_loss(current_price, current_time):
                continue
            
            # Get the latest H1 trend
            h1_row = h1_data[h1_data.index <= current_time].iloc[-1]
            trend_bias = h1_row['trend_bias']
            
            # Current M5 row
            m5_row = m5_data.loc[idx]
            
            # Entry conditions with RSI confirmation
            if self.current_trade is None:  # No open position
                if trend_bias == 'bullish' and m5_row['price_crossed_above_ema'] and m5_row['rsi_9'] > 30:
                    self.open_trade(current_price, 'buy', current_time)
                elif trend_bias == 'bearish' and m5_row['price_crossed_below_ema'] and m5_row['rsi_9'] < 70:
                    self.open_trade(current_price, 'sell', current_time)
            
            # Exit conditions based on MACD crossovers
            elif self.current_trade['direction'] == 'buy':
                if m5_row['macd_cross_below']:
                    self.close_trade(current_price, current_time, "signal")
            elif self.current_trade['direction'] == 'sell':
                if m5_row['macd_cross_above']:
                    self.close_trade(current_price, current_time, "signal")
        
        return self.trades_history

In [47]:
def prepare_data(m5_data, h1_data):
    """Prepare market data with technical indicators"""
    # H1 data preparation
    h1_data['ema8'] = ta.ema(h1_data['close'], length=8)
    h1_data['trend_bias'] = np.where(h1_data['close'] > h1_data['ema8'], 'bullish', 'bearish')
    
    # M5 data preparation
    m5_data['ema5'] = ta.ema(m5_data['close'], length=5)
    # Add safety check for empty DataFrames
    if len(m5_data) == 0 or len(h1_data) == 0:
        raise ValueError("Empty DataFrame detected")

    # Use boolean indexing instead of direct index access
    m5_data['price_crossed_above_ema'] = (
        (m5_data['close'] > m5_data['ema5']) & 
        (m5_data['close'].shift(1) < m5_data['ema5'].shift(1))
    )
    m5_data['price_crossed_below_ema'] = (
        (m5_data['close'] < m5_data['ema5']) & 
        (m5_data['close'].shift(1) > m5_data['ema5'].shift(1))
    )
    
    # RSI and MACD
    m5_data['rsi_9'] = ta.rsi(m5_data['close'], length=9)
    macd = ta.macd(m5_data['close'], fast=8, slow=17, signal=9)
    m5_data['macd_line'] = macd['MACD_8_17_9']
    m5_data['macd_signal'] = macd['MACDs_8_17_9']
    m5_data['macd_cross_below'] = (
        (m5_data['macd_line'] < m5_data['macd_signal']) & 
        (m5_data['macd_line'].shift(1) > m5_data['macd_signal'].shift(1))
    )
    m5_data['macd_cross_above'] = (
        (m5_data['macd_line'] > m5_data['macd_signal']) & 
        (m5_data['macd_line'].shift(1) < m5_data['macd_signal'].shift(1))
    )
    
    return m5_data, h1_data

In [48]:
# Load trading data
df = pd.read_csv('trades_history.csv')

# Convert datetime columns
df['entry_time'] = pd.to_datetime(df['entry_time'])
df['exit_time'] = pd.to_datetime(df['exit_time'])

# Add session information
def get_trading_session(timestamp):
    hour = timestamp.hour
    if 16 <= hour < 20:  # London-NY overlap (16:00-20:00)
        return 'London-NY'
    return 'Outside Session'

df['trading_session'] = df['entry_time'].apply(get_trading_session)

# Filter and analyze sessions
session_trades = df.groupby('trading_session').agg({
    'profit': ['count', 'sum', 'mean', 'std'],
    'pips': ['mean', 'std']
}).round(2)

print("\nTrading Session Analysis:")
print(session_trades)

# Calculate win rates per session
def calculate_session_metrics(session_data):
    total_trades = len(session_data)
    winning_trades = len(session_data[session_data['profit'] > 0])
    win_rate = (winning_trades / total_trades * 100) if total_trades > 0 else 0
    profit_factor = abs(session_data[session_data['profit'] > 0]['profit'].sum() / 
                       session_data[session_data['profit'] < 0]['profit'].sum()) if len(session_data[session_data['profit'] < 0]) > 0 else float('inf')
    return pd.Series({
        'Total Trades': total_trades,
        'Win Rate %': round(win_rate, 2),
        'Profit Factor': round(profit_factor, 2),
        'Net Profit': round(session_data['profit'].sum(), 2),
        'Avg Profit': round(session_data['profit'].mean(), 2),
        'Avg Pips': round(session_data['pips'].mean(), 2)
    })

session_metrics = df.groupby('trading_session').apply(calculate_session_metrics)
print("\nDetailed Session Metrics:")
print(session_metrics)


Trading Session Analysis:
                profit                     pips       
                 count    sum  mean   std  mean    std
trading_session                                       
London-NY           42  14.39  0.34  1.67  3.43  16.70
Outside Session    180  27.20  0.15  1.49  1.51  14.91

Detailed Session Metrics:
                 Total Trades  Win Rate %  Profit Factor  Net Profit  \
trading_session                                                        
London-NY                42.0       54.76           1.83       14.39   
Outside Session         180.0       47.78           1.46       27.20   

                 Avg Profit  Avg Pips  
trading_session                        
London-NY              0.34      3.43  
Outside Session        0.15      1.51  


/tmp/ipykernel_33583/395855119.py:42: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [49]:
# Visualize profit distribution by session
fig = px.box(df, x='trading_session', y='profit',
             title='Profit Distribution by Trading Session')
fig.show()

# Cumulative profit by session
df['cumulative_profit'] = df.groupby('trading_session')['profit'].cumsum()
fig = px.line(df, x='entry_time', y='cumulative_profit', color='trading_session',
              title='Cumulative Profit by Trading Session')
fig.show()

# Win rate analysis
session_win_rates = df.groupby(['trading_session', df['profit'] > 0]).size().unstack()
session_win_rates['win_rate'] = session_win_rates[True] / session_win_rates.sum(axis=1) * 100
print("\nWin Rates by Session:")
print(session_win_rates['win_rate'].round(2))


Win Rates by Session:
trading_session
London-NY          54.76
Outside Session    47.78
Name: win_rate, dtype: float64


In [50]:
# Analyze close reasons by session
close_reason_analysis = pd.crosstab(df['trading_session'], df['close_reason'])
print("\nClose Reasons by Session:")
print(close_reason_analysis)

# Average profit by close reason and session
avg_profit_by_reason = df.groupby(['trading_session', 'close_reason'])['profit'].agg(['mean', 'count']).round(2)
print("\nAverage Profit by Close Reason and Session:")
print(avg_profit_by_reason)

# Plot average profit by close reason and session
fig = px.bar(df, x='trading_session', y='profit', color='close_reason',
             title='Average Profit by Session and Close Reason',
             barmode='group')
fig.show()


Close Reasons by Session:
close_reason     signal  stop_loss
trading_session                   
London-NY            40          2
Outside Session     176          4

Average Profit by Close Reason and Session:
                              mean  count
trading_session close_reason             
London-NY       signal        0.51     40
                stop_loss    -3.00      2
Outside Session signal        0.22    176
                stop_loss    -2.68      4


## Summary of Trading Performance
- Analysis of trading performance during London-NY session (16:00-20:00) vs outside session
- Comparison of win rates, profit factors, and average profits
- Distribution of stop-loss vs signal-based exits
- Cumulative profit trends by session

In [51]:
# Calculate Financial Metrics
def calculate_financial_metrics(df):
    metrics = {}
    
    # Basic Metrics
    metrics['Total Trades'] = len(df)
    metrics['Winning Trades'] = len(df[df['profit'] > 0])
    metrics['Losing Trades'] = len(df[df['profit'] < 0])
    metrics['Win Rate'] = (metrics['Winning Trades'] / metrics['Total Trades']) * 100
    
    # Profit Metrics
    metrics['Total Profit'] = df['profit'].sum()
    metrics['Average Profit'] = df['profit'].mean()
    metrics['Max Profit'] = df['profit'].max()
    metrics['Max Loss'] = df['profit'].min()
    
    # Risk Metrics
    profits = df[df['profit'] > 0]['profit']
    losses = abs(df[df['profit'] < 0]['profit'])
    metrics['Average Win'] = profits.mean() if len(profits) > 0 else 0
    metrics['Average Loss'] = losses.mean() if len(losses) > 0 else 0
    metrics['Profit Factor'] = (profits.sum() / losses.sum()) if len(losses) > 0 else float('inf')
    metrics['Risk Reward Ratio'] = metrics['Average Win'] / metrics['Average Loss'] if metrics['Average Loss'] != 0 else float('inf')
    
    # Drawdown Analysis
    df['cumulative'] = df['profit'].cumsum()
    df['peak'] = df['cumulative'].cummax()
    df['drawdown'] = df['peak'] - df['cumulative']
    metrics['Max Drawdown'] = df['drawdown'].max()
    metrics['Max Drawdown %'] = (metrics['Max Drawdown'] / df['peak'].max()) * 100 if df['peak'].max() != 0 else 0
    
    # Trade Duration
    df['duration'] = pd.to_datetime(df['exit_time']) - pd.to_datetime(df['entry_time'])
    metrics['Average Duration'] = df['duration'].mean()
    metrics['Max Duration'] = df['duration'].max()
    metrics['Min Duration'] = df['duration'].min()
    
    # Performance Ratios
    trading_days = (df['exit_time'].max() - df['entry_time'].min()).days + 1
    metrics['Profit per Day'] = metrics['Total Profit'] / trading_days if trading_days > 0 else 0
    metrics['Trades per Day'] = metrics['Total Trades'] / trading_days if trading_days > 0 else 0
    metrics['Sharpe Ratio'] = (df['profit'].mean() / df['profit'].std()) * np.sqrt(252) if df['profit'].std() != 0 else 0
    
    return pd.Series(metrics)

# Calculate metrics for both sessions
london_ny = df[df['trading_session'] == 'London-NY']
outside = df[df['trading_session'] == 'Outside Session']

metrics_london_ny = calculate_financial_metrics(london_ny)
metrics_outside = calculate_financial_metrics(outside)

# Create comparison DataFrame
comparison = pd.DataFrame({
    'London-NY Session': metrics_london_ny,
    'Outside Session': metrics_outside
}).round(2)

print("\nDetailed Financial Metrics Comparison:")
print(comparison)

# Visualize Key Metrics
fig = go.Figure()
fig.add_trace(go.Bar(
    name='London-NY',
    x=['Win Rate', 'Profit Factor', 'Risk Reward Ratio', 'Sharpe Ratio'],
    y=[metrics_london_ny['Win Rate'], 
       metrics_london_ny['Profit Factor'],
       metrics_london_ny['Risk Reward Ratio'],
       metrics_london_ny['Sharpe Ratio']]
))
fig.add_trace(go.Bar(
    name='Outside Session',
    x=['Win Rate', 'Profit Factor', 'Risk Reward Ratio', 'Sharpe Ratio'],
    y=[metrics_outside['Win Rate'],
       metrics_outside['Profit Factor'],
       metrics_outside['Risk Reward Ratio'],
       metrics_outside['Sharpe Ratio']]
))
fig.update_layout(title='Key Performance Metrics by Session',
                 barmode='group')
fig.show()

# Plot equity curve
fig = go.Figure()
for session in ['London-NY', 'Outside Session']:
    session_data = df[df['trading_session'] == session]
    fig.add_trace(go.Scatter(
        x=session_data['exit_time'],
        y=session_data['profit'].cumsum(),
        name=f'{session} Equity',
        mode='lines'
    ))
fig.update_layout(title='Equity Curves by Trading Session',
                 xaxis_title='Time',
                 yaxis_title='Cumulative Profit')
fig.show()


Detailed Financial Metrics Comparison:
                           London-NY Session            Outside Session
Total Trades                              42                        180
Winning Trades                            23                         86
Losing Trades                             19                         92
Win Rate                           54.761905                  47.777778
Total Profit                           14.39                       27.2
Average Profit                      0.342619                   0.151111
Max Profit                              5.85                      13.47
Max Loss                               -3.47                       -2.8
Average Win                         1.380435                   0.997326
Average Loss                        0.913684                    0.63663
Profit Factor                       1.828917                   1.464402
Risk Reward Ratio                   1.510845                   1.566569
Max Drawdown            

/tmp/ipykernel_33583/249196951.py:26: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_33583/249196951.py:27: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_33583/249196951.py:28: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_3

## Financial Metrics Analysis

The above analysis provides detailed financial metrics comparing trading performance between London-NY session and outside session trading:

1. **Profitability Metrics**
   - Total and average profits
   - Win rates and profit factors
   - Maximum profit and loss

2. **Risk Metrics**
   - Risk-reward ratios
   - Maximum drawdown
   - Sharpe ratio

3. **Trading Efficiency**
   - Average trade duration
   - Trades per day
   - Profit per day

4. **Visualizations**
   - Key performance metrics comparison
   - Equity curves by session

In [52]:
def get_state(data, m5_idx, h1_data):
    """Create state vector from indicators with safety checks"""
    try:
        # Use loc instead of direct indexing
        current_m5 = data.loc[m5_idx]
        # Find the most recent H1 data point
        current_h1 = h1_data[h1_data.index <= m5_idx].iloc[-1] if len(h1_data) > 0 else None
        
        if current_h1 is None:
            raise ValueError("No valid H1 data found for the given timestamp")
            
        return np.array([
            current_m5['close'],
            current_m5['ema5'],
            current_m5['rsi_9'],
            current_m5['macd_line'],
            current_m5['macd_signal'],
            current_h1['ema8'],
            current_h1['trend_bias'] == 'bullish'  # Convert to binary
        ])
    except KeyError as e:
        logger.error(f"Failed to access data at index {m5_idx}: {str(e)}")
        raise

In [53]:
class Action:
    HOLD = 0
    BUY = 1
    SELL = 2

In [54]:
class TradingEnv:
    def __init__(self, m5_data, h1_data, initial_balance=10000):
        self.simulator = TradingSimulator(initial_balance)
        self.m5_data = m5_data
        self.h1_data = h1_data
        self.current_step = 0
        self.index_list = list(m5_data.index)  # Store index values as a list
        self.max_steps = len(m5_data) - 1
        
    def reset(self):
        self.current_step = 0
        self.simulator = TradingSimulator()
        return get_state(self.m5_data, self.index_list[0], self.h1_data)
        
    def step(self, action):
        if self.current_step >= len(self.index_list):
            raise IndexError("Episode finished - reset environment")
            
        current_time = self.index_list[self.current_step]
        current_price = self.m5_data.loc[current_time, 'close']
        done = self.current_step >= self.max_steps
        
        # Execute action
        if action == Action.BUY and not self.simulator.current_trade:
            self.simulator.open_trade(current_price, 'buy', current_time)
        elif action == Action.SELL and not self.simulator.current_trade:
            self.simulator.open_trade(current_price, 'sell', current_time)
            
        # Check stop loss
        self.simulator.check_stop_loss(current_price, current_time)
        
        # Get reward
        reward = self._calculate_reward(current_price)
        
        # Move to next step
        self.current_step += 1
        
        # Get next state
        if not done:
            next_state = get_state(self.m5_data, self.index_list[self.current_step], self.h1_data)
        else:
            next_state = get_state(self.m5_data, self.index_list[-1], self.h1_data)
            
        return next_state, reward, done, {}

    def _calculate_reward(self, current_price):
        if not self.simulator.current_trade:
            return 0  # No reward for holding
            
        unrealized_pnl = (current_price - self.simulator.current_trade['entry_price']) * 10000
        if self.simulator.current_trade['direction'] == 'sell':
            unrealized_pnl *= -1
            
        return unrealized_pnl * self.simulator.pip_value

In [56]:
class ReplayBuffer:
    """Experience replay buffer for storing and sampling transitions"""
    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        if len(self.buffer) < batch_size:
            return None
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)

class DQNAgent:
    """Deep Q-Network agent for trading"""
    def __init__(self, env, initial_policy=True):
        self.env = env
        self.initial_policy = initial_policy
        self.buffer = ReplayBuffer()
        self.batch_size = 64
        self.gamma = 0.95  # Discount factor
        self.epsilon = 1.0  # Initial exploration rate
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.target_update_freq = 100  # Update target network every 100 steps
        self.step_count = 0
        
        # Initialize both main and target networks
        self.model = self._build_model()
        self.target_model = self._build_model()
        self.target_model.set_weights(self.model.get_weights())

    def _build_model(self):
        """Create neural network model with batch normalization"""
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(7,)),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dense(3)  # Three actions: HOLD, BUY, SELL
        ])
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                      loss='mse')
        return model

    def update_target_network(self):
        """Update target network weights with main network weights"""
        self.target_model.set_weights(self.model.get_weights())

    def get_action(self, state, training=True):
        """Epsilon-greedy strategy with decay"""
        if training and np.random.random() < self.epsilon:
            return np.random.choice([Action.HOLD, Action.BUY, Action.SELL])
        
        if self.initial_policy:
            return self._initial_policy_action(state)
        
        q_values = self.model.predict(state[np.newaxis], verbose=0)
        return np.argmax(q_values[0])

    def _initial_policy_action(self, state):
        """Your existing strategy logic"""
        price_above_ema = state[0] > state[1]
        rsi = state[2]
        trend_bullish = state[6]
        
        if trend_bullish and price_above_ema and rsi > 30:
            return Action.BUY
        elif not trend_bullish and not price_above_ema and rsi < 70:
            return Action.SELL
        return Action.HOLD

    def train(self, episodes=1000):
        """Training loop with experience replay"""
        for episode in range(episodes):
            state = self.env.reset()
            total_reward = 0
            done = False
            
            while not done:
                action = self.get_action(state)
                next_state, reward, done, _ = self.env.step(action)
                self.buffer.add(state, action, reward, next_state, done)
                total_reward += reward
                state = next_state
                self.step_count += 1

                # Train on batch from replay buffer
                if len(self.buffer) >= self.batch_size:
                    self._train_on_batch()

                # Update target network periodically
                if self.step_count % self.target_update_freq == 0:
                    self.update_target_network()

            # Decay epsilon after each episode
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

            print(f"Episode {episode+1}/{episodes} | Total Reward: {total_reward:.2f} | Epsilon: {self.epsilon:.3f}")

    def _train_on_batch(self):
        """Batch training logic with target network"""
        batch = self.buffer.sample(self.batch_size)
        if not batch:
            return

        states, actions, rewards, next_states, dones = zip(*batch)
        
        # Convert to numpy arrays
        states = np.array(states)
        next_states = np.array(next_states)
        
        # Predict Q-values using main and target networks
        current_q = self.model.predict(states, verbose=0)
        next_q = self.target_model.predict(next_states, verbose=0)
        
        # Calculate target Q-values
        targets = np.copy(current_q)
        for i in range(len(batch)):
            if dones[i]:
                targets[i, actions[i]] = rewards[i]
            else:
                targets[i, actions[i]] = rewards[i] + self.gamma * np.max(next_q[i])

        # Train the model
        self.model.train_on_batch(states, targets)

    def save_model(self, path='dqn_model'):
        """Save both main and target networks"""
        self.model.save(f'{path}_main.h5')
        self.target_model.save(f'{path}_target.h5')
        print(f"Models saved to {path}_main.h5 and {path}_target.h5")

    def load_model(self, path='dqn_model'):
        """Load both main and target networks"""
        if os.path.exists(f'{path}_main.h5'):
            self.model = tf.keras.models.load_model(f'{path}_main.h5')
            self.target_model = tf.keras.models.load_model(f'{path}_target.h5')
            print("Models loaded successfully")
        else:
            print("No saved models found")

# Modified run_rl_training function
def run_rl_training(resume_training=False):
    """Run the complete RL training pipeline with error handling"""
    try:
        # Load and prepare data
        m5_data = pd.read_csv('M5.csv', index_col=0, parse_dates=True)
        h1_data = pd.read_csv('H1.csv', index_col=0, parse_dates=True)
        
        # Validate data
        if m5_data.empty or h1_data.empty:
            raise ValueError("Empty data files")
            
        # Sort indices to ensure proper time alignment
        m5_data = m5_data.sort_index()
        h1_data = h1_data.sort_index()
        
        # Prepare data with indicators
        m5_data, h1_data = prepare_data(m5_data, h1_data)
        
        # Create environment and agent
        env = TradingEnv(m5_data, h1_data)
        agent = DQNAgent(env, initial_policy=True)
        
        # Training logic
        if resume_training:
            agent.load_model()
            print("Resuming training with loaded model...")
            agent.initial_policy = False
        else:
            print("Starting new training session...")
            print("Phase 1: Warm-start with initial policy")
            agent.train(episodes=100)
            agent.save_model()
        
        print("Phase 2: Full training with exploration")
        agent.initial_policy = False
        agent.train(episodes=1000)
        agent.save_model()
        
    except Exception as e:
        logger.error(f"Training failed: {str(e)}", exc_info=True)
        raise

if __name__ == '__main__':
    run_rl_training(resume_training=False)

/home/chiqo/Documents/AI/High-Frequency-Trading-Model-with-IB/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.

2025-05-15 18:11:56,853 - INFO - Opened sell position: Entry=1.13459, SL=1.13709, Size=0.01
2025-05-15 18:11:56,925 - INFO - Closed trade: Pips=-26.2, Profit=$-2.62, Balance=$97.38
2025-05-15 18:11:56,931 - INFO - Opened buy position: Entry=1.13691, SL=1.1344100000000001, Size=0.01
2025-05-15 18:11:56,925 - INFO - Closed trade: Pips=-26.2, Profit=$-2.62, Balance=$97.38
2025-05-15 18:11:56,931 - INFO - Opened buy position: Entry=1.13691, SL=1.1344100000000001, Size=0.01


Starting new training session...
Phase 1: Warm-start with initial policy


KeyboardInterrupt: 